# 1. 讀取資料
---
## 資料來源
1. 河川資料：水利署
2. 避難收容所資料：消防署（政府開放平台）
3. 鄉鎮市區界：國土測繪中心

## 輸出結果
1. 共 13262 筆河川資料；CRS: EPSG:3826
2. 共 5973 筆避難所資料，並轉換為 GeoDataFrame 格式
3. 共 368 筆鄉鎮資料

In [ ]:
import geopandas as gpd

# 讀取本地的河川資料
rivers = gpd.read_file('Data/riverpoly/riverpoly.shp')
print(f"河川資料載入完成，共 {len(rivers)} 筆")
print(f"CRS: {rivers.crs}")

import pandas as pd

# 讀取避難所資料
shelters_csv = pd.read_csv('Data/避難收容處所點位檔案v9.csv')
print(f"避難所資料載入完成，共 {len(shelters_csv)} 筆")

# 轉換為 GeoDataFrame
shelters = gpd.GeoDataFrame(
    shelters_csv,
    geometry=gpd.points_from_xy(shelters_csv['經度'], shelters_csv['緯度']),
    crs='EPSG:4326'
)
print(f"避難所 GeoDataFrame 資料載入完成，共 {len(shelters)} 筆")

# 讀取鄉鎮資料 - 指定正確的圖層
from urllib.parse import quote
url = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/' + quote('鄉(鎮、市、區)界線1140318.zip')
townships = gpd.read_file(url, layer='TOWN_MOI_1140318')
print(f"鄉鎮資料載入完成，共 {len(townships)} 筆")

# 2. 確認CRS
所有資料皆應投影至EPSG3826

In [ ]:
# CRS 轉換 - 確保所有資料都是 EPSG:3826

# 1. 確認 rivers 是否為 EPSG:3826

rivers = rivers.to_crs('EPSG:3826')
print(f"河川資料已轉換為 EPSG:3826")

# 2. 將 shelter 轉換為 EPSG:3826
shelters = shelters.to_crs('EPSG:3826')
print(f"避難所轉換後 CRS: {shelters.crs}")

# 3. 將 townships 轉換為 EPSG:3826
townships = townships.to_crs('EPSG:3826')
print(f"鄉鎮資料已轉換為 EPSG:3826")

print("\n所有資料都已轉換為 EPSG:3826")

# 3. 資料清理
---
## 避難所資料清理
1. 重新建立 GeoDataFrame
2. 座標範圍檢查
3. 檢查空職或經緯度為零的資料
4. 比對鄉鎮邊界資料移除剩餘不合理點位

In [ ]:
# 避難所資料清理 - 結合座標範圍和鄉鎮界線檢查

print("=== 避難所資料清理 ===")

# 重新建立 GeoDataFrame
shelters_original = gpd.GeoDataFrame(
    shelters_csv,
    geometry=gpd.points_from_xy(shelters_csv['經度'], shelters_csv['緯度']),
    crs='EPSG:4326'
)

# 轉換為 EPSG:3826
shelters_original = shelters_original.to_crs('EPSG:3826')

print(f"重新建立 GeoDataFrame，共 {len(shelters_original)} 筆")

original_count = len(shelters_original)

# === 第一步：座標範圍檢查 ===
print(f"\n=== 第一步：座標範圍檢查 ===")

# 1. 檢查空值和零值的筆數
null_lon = shelters_original['經度'].isna().sum()
null_lat = shelters_original['緯度'].isna().sum()
zero_lon = (shelters_original['經度'] == 0).sum()
zero_lat = (shelters_original['緯度'] == 0).sum()

print(f"經度為空值: {null_lon} 筆")
print(f"緯度為空值: {null_lat} 筆")
print(f"經度為 0: {zero_lon} 筆")
print(f"緯度為 0: {zero_lat} 筆")

# 2. 檢查超出台灣範圍的筆數 (經度 119~122, 緯度 21~26)
out_of_range_condition = (
    (shelters_original['經度'] < 119) | (shelters_original['經度'] > 122) |
    (shelters_original['緯度'] < 21) | (shelters_original['緯度'] > 26)
)

out_of_range_count = out_of_range_condition.sum()
print(f"超出台灣範圍(經度119~122, 緯度21~26): {out_of_range_count} 筆")

# 3. 座標範圍清理
range_clean_condition = (
    shelters_original['經度'].notna() &
    shelters_original['緯度'].notna() &
    (shelters_original['經度'] != 0) &
    (shelters_original['緯度'] != 0) &
    (shelters_original['經度'] >= 119) &
    (shelters_original['經度'] <= 122) &
    (shelters_original['緯度'] >= 21) &
    (shelters_original['緯度'] <= 26)
)

shelters_step1 = shelters_original[range_clean_condition]
range_removed = original_count - len(shelters_step1)

print(f"座標範圍清理後剩餘: {len(shelters_step1)} 筆")
print(f"座標範圍移除: {range_removed} 筆")

# === 第二步：鄉鎮界線檢查 ===
print(f"\n=== 第二步：鄉鎮界線檢查 ===")

# 確保 CRS 一致
if shelters_step1.crs != townships.crs:
    print("CRS 不一致，正在轉換...")
    shelters_step1 = shelters_step1.to_crs(townships.crs)

# 使用 spatial join 找出在鄉鎮界線內的避難所
shelters_in_townships = gpd.sjoin(shelters_step1, townships, how='inner', predicate='within')
township_removed = len(shelters_step1) - len(shelters_in_townships)

print(f"在鄉鎮界線內: {len(shelters_in_townships)} 筆")
print(f"不在鄉鎮界線內: {township_removed} 筆")

# === 最終清理結果 ===
shelter_true = shelters_in_townships.copy()

# 移除 sjoin 產生的多餘欄位
if 'index_right' in shelter_true.columns:
    shelter_true = shelter_true.drop(columns=['index_right'])

final_count = len(shelter_true)
total_removed = original_count - final_count

print(f"\n=== 最終清理結果 ===")
print(f"清理前資料筆數: {original_count}")
print(f"清理後資料筆數: {final_count}")
print(f"總移除: {total_removed} 筆")
print(f"資料保留率: {final_count/original_count*100:.1f}%")

print(f"\n移除原因分析：")
print(f"- 座標範圍問題: {range_removed} 筆")
print(f"- 不在鄉鎮界線內: {township_removed} 筆")
print(f"- 總計: {total_removed} 筆")

# 檢查清理後的座標範圍
print(f"\n清理後座標範圍:")
print(f"經度範圍: {shelter_true['經度'].min():.6f} ~ {shelter_true['經度'].max():.6f}")
print(f"緯度範圍: {shelter_true['緯度'].min():.6f} ~ {shelter_true['緯度'].max():.6f}")

# 統計各縣市的避難所數量
if 'COUNTYNAME' in shelter_true.columns:
    print(f"\n各縣市避難所分布:")
    county_counts = shelter_true['COUNTYNAME'].value_counts().head(10)
    for county, count in county_counts.items():
        print(f"  - {county}: {count} 筆")

print(f"\n✓ 避難所資料清理完成，shelter_true 已更新為清理後的 {len(shelter_true)} 筆資料")
print("✓ 結合座標範圍檢查和鄉鎮界線檢查，確保資料品質")

# 4. 三級河川警戒緩衝區：
---
## 目標:建立三層緩衝區：高風險 / 中風險 / 低風險
### 流程:
1. 從.env讀取buffer範圍資料
2. 從高風險緩衝區開始建立(500公尺)
3. 中風險緩衝區建立(1000公尺)
4. 低風險緩衝區建立(2000公尺)
5. 檢查緩衝區面積是否合理

In [ ]:
# 三級河川警戒緩衝區分析 (正確的環形)

print("=== 三級河川警戒緩衝區分析 (正確環形) ===")

# 導入必要套件
from dotenv import load_dotenv
import os

# 讀取環境變數
load_dotenv()

BUFFER_HIGH = int(os.getenv('BUFFER_HIGH'))
BUFFER_MED = int(os.getenv('BUFFER_MED'))
BUFFER_LOW = int(os.getenv('BUFFER_LOW'))

print(f"緩衝距離: 高={BUFFER_HIGH}m, 中={BUFFER_MED}m, 低={BUFFER_LOW}m")

# 確認河川資料的 CRS
print(f"河川資料 CRS: {rivers.crs}")
print(f"河川資料筆數: {len(rivers)}")

# 建立緩衝區
print("\n建立環形緩衝區...")

# 高風險緩衝區 (0-500m)
buffer_high = rivers.buffer(BUFFER_HIGH)
print(f"✓ 高風險緩衝區 (0-{BUFFER_HIGH}m)")

# 中風險緩衝區 (500-1000m) - 環形
buffer_med = rivers.buffer(BUFFER_MED).difference(rivers.buffer(BUFFER_HIGH))
print(f"✓ 中風險緩衝區 ({BUFFER_HIGH}-{BUFFER_MED}m)")

# 低風險緩衝區 (1000-2000m) - 環形  
buffer_low = rivers.buffer(BUFFER_LOW).difference(rivers.buffer(BUFFER_MED))
print(f"✓ 低風險緩衝區 ({BUFFER_MED}-{BUFFER_LOW}m")

# 檢查幾何類型
print(f"\n幾何類型檢查:")
print(f"高風險區: {buffer_high.geom_type.iloc[0] if len(buffer_high) > 0 else '無'}")
print(f"中風險區: {buffer_med.geom_type.iloc[0] if len(buffer_med) > 0 else '無'}")
print(f"低風險區: {buffer_low.geom_type.iloc[0] if len(buffer_low) > 0 else '無'}")

# 建立統一的緩衝區 GeoDataFrame
buffer_zones_list = []

# 高風險緩衝區
buffer_high_zones = gpd.GeoDataFrame(geometry=buffer_high, crs=rivers.crs)
buffer_high_zones['buffer_level'] = '高'
buffer_high_zones['area_km2'] = buffer_high_zones.geometry.area / 1000000
buffer_zones_list.append(buffer_high_zones)

# 中風險緩衝區
buffer_med_zones = gpd.GeoDataFrame(geometry=buffer_med, crs=rivers.crs)
buffer_med_zones['buffer_level'] = '中'
buffer_med_zones['area_km2'] = buffer_med_zones.geometry.area / 1000000
buffer_zones_list.append(buffer_med_zones)

# 低風險緩衝區
buffer_low_zones = gpd.GeoDataFrame(geometry=buffer_low, crs=rivers.crs)
buffer_low_zones['buffer_level'] = '低'
buffer_low_zones['area_km2'] = buffer_low_zones.geometry.area / 1000000
buffer_zones_list.append(buffer_low_zones)

# 合併所有緩衝區
river_buffer_zones = pd.concat(buffer_zones_list, ignore_index=True)
print(f"\n✓ 統一緩衝區建立完成，共 {len(river_buffer_zones)} 個")

# 統計資訊
print(f"\n=== 緩衝區統計 ===")
print(f"高風險區: {len(buffer_high_zones)} 個 ({buffer_high_zones['area_km2'].sum():.2f} km²)")
print(f"中風險區: {len(buffer_med_zones)} 個 ({buffer_med_zones['area_km2'].sum():.2f} km²)")
print(f"低風險區: {len(buffer_low_zones)} 個 ({buffer_low_zones['area_km2'].sum():.2f} km²)")

print(f"\n✓ 環形緩衝區建立完成")

# 5. 空間連接（Spatial Join）
---
### 目標：將所有避難收容所標記風險程度
### 分類依據如下：  
    若避難收容所位於河川警戒緩衝區內，則視為具有風險，並依所在緩衝區範圍判定其風險程度。  
    若同一避難收容所同時落在多個風險區內，則以風險等級較高者作為最終分類結果。

### 流程:
1. 初始化避難收容所資料，全部先標記為「安全」
2. 標記位於高風險區的避難收容所
3. 排除已標記為高風險者後，標記中風險避難收容所
4. 排除已標記為高風險與中風險者後，標記低風險避難收容所
5. 排除已標記為高、中、低風險者後，其餘皆歸類為安全避難收容所

### 輸出結果
輸出 `shelter_marked.csv`，內容包含各避難收容所之風險程度欄位。

In [ ]:
# 空間連接分析 - 標記避難所風險等級

print("=== 空間連接分析 - 標記避難所風險等級 ===")

# 複製避難所資料以避免修改原始資料
shelters_with_risk = shelter_true.copy()

# 初始化風險等級欄位
shelters_with_risk['risk_level'] = '安全'  # 預設為安全

print(f"開始分析 {len(shelters_with_risk)} 筆避難所的風險等級...")

# 1. 找出高風險緩衝區內的避難所 (最高優先級)
high_risk_join = gpd.sjoin(shelters_with_risk, buffer_high_zones, how='inner', predicate='within')
high_risk_indices = high_risk_join.index.unique()

# 標記高風險
shelters_with_risk.loc[high_risk_indices, 'risk_level'] = '高'

# 2. 找出中風險緩衝區內的避難所 (排除已標記為高風險的)
remaining_shelters = shelters_with_risk[shelters_with_risk['risk_level'] == '安全']
med_risk_join = gpd.sjoin(remaining_shelters, buffer_med_zones, how='inner', predicate='within')
med_risk_indices = med_risk_join.index.unique()

# 標記中風險
shelters_with_risk.loc[med_risk_indices, 'risk_level'] = '中'

# 3. 找出低風險緩衝區內的避難所 (排除已標記為高、中風險的)
remaining_shelters = shelters_with_risk[shelters_with_risk['risk_level'] == '安全']
low_risk_join = gpd.sjoin(remaining_shelters, buffer_low_zones, how='inner', predicate='within')
low_risk_indices = low_risk_join.index.unique()

# 標記低風險
shelters_with_risk.loc[low_risk_indices, 'risk_level'] = '低'

# 4. 統計各風險等級的避難所數量
print(f"\n=== 風險等級統計 ===")
risk_counts = shelters_with_risk['risk_level'].value_counts()
print(f"高風險避難所: {risk_counts.get('高', 0)} 筆")
print(f"中風險避難所: {risk_counts.get('中', 0)} 筆")
print(f"低風險避難所: {risk_counts.get('低', 0)} 筆")
print(f"安全避難所: {risk_counts.get('安全', 0)} 筆")
print(f"總計: {risk_counts.sum()} 筆")

# 5. 計算風險覆蓋率
total_shelters = len(shelters_with_risk)
high_risk_percentage = risk_counts.get('高', 0) / total_shelters * 100
med_risk_percentage = risk_counts.get('中', 0) / total_shelters * 100
low_risk_percentage = risk_counts.get('低', 0) / total_shelters * 100
safe_percentage = risk_counts.get('安全', 0) / total_shelters * 100

print(f"\n=== 風險覆蓋率 ===")
print(f"高風險區域: {high_risk_percentage:.1f}%")
print(f"中風險區域: {med_risk_percentage:.1f}%")
print(f"低風險區域: {low_risk_percentage:.1f}%")
print(f"安全區域: {safe_percentage:.1f}%")

# 6. 檢查重複標記問題
print(f"\n=== 資料完整性檢查 ===")
duplicates = shelters_with_risk.index.duplicated().sum()
if duplicates == 0:
    print("✓ 沒有發現重複標記問題")
else:
    print(f"⚠ 發現 {duplicates} 筆重複標記，需要處理")

# 準備輸出的資料 (移除幾何欄位，保留原始欄位 + 風險等級)
output_data = shelters_with_risk.drop(columns=['geometry'])

# 確保經緯度欄位存在
if '經度' not in output_data.columns:
    output_data['經度'] = shelters_with_risk.geometry.x
if '緯度' not in output_data.columns:
    output_data['緯度'] = shelters_with_risk.geometry.y

# 重新排列欄位順序 (將風險等級放在最後)
cols = list(output_data.columns)
if 'risk_level' in cols:
    cols.remove('risk_level')
    cols.append('risk_level')
output_data = output_data[cols]

# 建立輸出資料夾
output_folder = 'output'
os.makedirs(output_folder, exist_ok=True)

# 設定輸出檔案完整路徑
output_filename = os.path.join(output_folder, 'shelter_marked.csv')

# 輸出 CSV
output_data.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"\n已輸出風險標記檔案: {output_filename}")


# 6. 收容量缺口分析 (Capacity Gap Analysis)
---
### 目標：統計避難所收容人數與風險分布，並輸出分析結果

1. 檢查避難所資料中的收容人數欄位
2. 將收容人數欄位轉為數值型態
3. 處理缺失值與異常值（如 0 值），以中位數或平均值補齊
4. 透過空間連接判定各避難所所屬鄉鎮市區與縣市
5. 統計各縣市避難所數量、總收容人數及各風險等級分布
6. 統計各風險等級的避難所數量、總收容人數與百分比
7. 輸出分析結果為 CSV 檔案

### 輸出結果
1. `各縣市避難所統計.csv`：包含各縣市避難所數量、總收容人數，以及高、中、低風險與安全避難所的詳細統計
2. `風險等級統計.csv`：包含各風險等級之避難所數量、總收容人數與百分比

In [ ]:
# 檢查收容人數欄位
print("\n檢查避難所資料欄位...")
print(f"可用欄位: {list(shelters_with_risk.columns)}")

# 找出收容人數相關欄位 - 更精確的匹配
capacity_cols = []
for col in shelters_with_risk.columns:
    col_lower = str(col).lower()
    if any(keyword in col_lower for keyword in ['收容人數', '容納人數', '人數', 'capacity']) and '地址' not in col and '名稱' not in col:
        capacity_cols.append(col)

print(f"收容人數相關欄位: {capacity_cols}")

if not capacity_cols:
    print("⚠ 未找到收容人數欄位，使用預設值 100")
    capacity_col = '收容人數'
    shelters_with_risk[capacity_col] = 100
else:
    # 選擇最可能的收容人數欄位
    capacity_col = capacity_cols[0]
    print(f"✓ 使用收容人數欄位: {capacity_col}")

# 檢查收容人數資料
print(f"\n收容人數資料檢查:")
print(f"欄位類型: {shelters_with_risk[capacity_col].dtype}")
print(f"前5筆資料: {shelters_with_risk[capacity_col].head().tolist()}")

# 處理收容人數資料 - 確保為數字類型
try:
    shelters_with_risk[capacity_col] = pd.to_numeric(shelters_with_risk[capacity_col], errors='coerce')
    print("✓ 收容人數欄位轉換為數字類型")
except Exception as e:
    print(f"❌ 收容人數欄位轉換失敗: {e}")
    raise

# 處理收容人數為0的異常值 - 保持整數類型
shelters_with_risk_clean = shelters_with_risk.copy()
zero_capacity_count = (shelters_with_risk_clean[capacity_col] == 0).sum()
null_capacity_count = shelters_with_risk_clean[capacity_col].isna().sum()

if null_capacity_count > 0:
    median_capacity = int(shelters_with_risk_clean[capacity_col].median())
    shelters_with_risk_clean.loc[shelters_with_risk_clean[capacity_col].isna(), capacity_col] = median_capacity
    print(f"✓ 將 {null_capacity_count} 筆缺失值填補為整數中位數 {median_capacity}")

if zero_capacity_count > 0:
    avg_capacity = int(shelters_with_risk_clean[capacity_col].mean())
    zero_mask = shelters_with_risk_clean[capacity_col] == 0
    shelters_with_risk_clean.loc[zero_mask, capacity_col] = avg_capacity
    print(f"✓ 將 {zero_capacity_count} 筆零值修正為整數平均值 {avg_capacity}")

# 確保收容人數欄位保持整數類型
shelters_with_risk_clean[capacity_col] = shelters_with_risk_clean[capacity_col].astype(int)
print(f"✓ 收容人數欄位保持為整數類型")

# 按鄉鎮市區進行空間連接，確定每個避難所所屬的鄉鎮
print(f"\n=== 鄉鎮市區空間連接 ===")
shelters_with_township = gpd.sjoin(shelters_with_risk_clean, townships, how='left', predicate='within')

# 確定正確的鄉鎮欄位名稱
township_cols = [col for col in shelters_with_township.columns if any(keyword in col.upper() for keyword in ['TOWN', 'COUNTY'])]
town_name_col = None
county_name_col = None

for col in township_cols:
    if 'TOWN' in col.upper() and 'NAME' in col.upper() and 'right' not in col.lower():
        town_name_col = col
    elif 'COUNTY' in col.upper() and 'NAME' in col.upper() and 'right' not in col.lower():
        county_name_col = col

if town_name_col != 'TOWNNAME':
    shelters_with_township['TOWNNAME'] = shelters_with_township[town_name_col]
if county_name_col != 'COUNTYNAME':
    shelters_with_township['COUNTYNAME'] = shelters_with_township[county_name_col]

# 移除無法匹配的記錄
shelters_with_township = shelters_with_township.dropna(subset=['TOWNNAME'])
print(f"匹配後避難所數量: {len(shelters_with_township)} 筆")

# === 第一個輸出檔案：各縣市統計 ===
print(f"\n=== 各縣市統計分析 ===")

# 按縣市分組統計
county_stats = shelters_with_township.groupby('COUNTYNAME').agg({
    capacity_col: 'sum',
    'risk_level': 'count'
}).round(1)

# 重新命名欄位
county_stats.columns = ['總收容人數', '避難所數量']
county_stats = county_stats.reset_index()

# 計算各風險等級的避難所數量和收容人數
print("計算各風險等級統計...")

# 高風險統計
high_risk_stats = shelters_with_township[shelters_with_township['risk_level'] == '高'].groupby('COUNTYNAME').agg({
    capacity_col: 'sum',
    'risk_level': 'count'
}).round(1)
high_risk_stats.columns = ['高風險收容人數', '高風險避難所數量']
high_risk_stats = high_risk_stats.reset_index()

# 中風險統計
med_risk_stats = shelters_with_township[shelters_with_township['risk_level'] == '中'].groupby('COUNTYNAME').agg({
    capacity_col: 'sum',
    'risk_level': 'count'
}).round(1)
med_risk_stats.columns = ['中風險收容人數', '中風險避難所數量']
med_risk_stats = med_risk_stats.reset_index()

# 低風險統計
low_risk_stats = shelters_with_township[shelters_with_township['risk_level'] == '低'].groupby('COUNTYNAME').agg({
    capacity_col: 'sum',
    'risk_level': 'count'
}).round(1)
low_risk_stats.columns = ['低風險收容人數', '低風險避難所數量']
low_risk_stats = low_risk_stats.reset_index()

# 安全區統計
safe_risk_stats = shelters_with_township[shelters_with_township['risk_level'] == '安全'].groupby('COUNTYNAME').agg({
    capacity_col: 'sum',
    'risk_level': 'count'
}).round(1)
safe_risk_stats.columns = ['安全收容人數', '安全避難所數量']
safe_risk_stats = safe_risk_stats.reset_index()

# 合併所有統計
county_stats = county_stats.merge(high_risk_stats, on='COUNTYNAME', how='left')
county_stats = county_stats.merge(med_risk_stats, on='COUNTYNAME', how='left')
county_stats = county_stats.merge(low_risk_stats, on='COUNTYNAME', how='left')
county_stats = county_stats.merge(safe_risk_stats, on='COUNTYNAME', how='left')

# 填補空值為0
county_stats = county_stats.fillna(0)

# 確保所有數量欄位為整數
numeric_cols = ['高風險收容人數', '高風險避難所數量', '中風險收容人數', '中風險避難所數量', 
                '低風險收容人數', '低風險避難所數量', '安全收容人數', '安全避難所數量']
for col in numeric_cols:
    if col in county_stats.columns:
        county_stats[col] = county_stats[col].astype(int)

# 重新排列欄位順序 - 保留所有原本欄位，然後加上安全避難所數量
county_stats = county_stats[['COUNTYNAME', '避難所數量', '總收容人數', '安全避難所數量', '安全收容人數', 
                          '高風險避難所數量', '高風險收容人數',
                          '中風險避難所數量', '中風險收容人數', 
                          '低風險避難所數量', '低風險收容人數']]
county_stats.columns = ['縣市名稱', '避難所數量', '總收容人數', '安全避難所數量', '安全收容人數',
                     '高風險避難所數量', '高風險收容人數',
                     '中風險避難所數量', '中風險收容人數',
                     '低風險避難所數量', '低風險收容人數']

print("各縣市統計:")
print(county_stats.to_string(index=False))

# 輸出第一個檔案
county_output = os.path.join(output_folder, '各縣市避難所統計.csv')
county_stats.to_csv(county_output, index=False, encoding='utf-8-sig')
print(f"\n✓ 已輸出各縣市統計檔案: {county_output}")

# === 第二個輸出檔案：風險等級統計 ===
print(f"\n=== 風險等級統計分析 ===")

# 統計各風險等級的避難所數量和收容人數
risk_stats = shelters_with_township.groupby('risk_level').agg({
    capacity_col: ['sum', 'count']
}).round(1)

# 重新命名欄位
risk_stats.columns = ['總收容人數', '避難所數量']
risk_stats = risk_stats.reset_index()
risk_stats.columns = ['風險等級', '總收容人數', '避難所數量']

# 確保收容人數為整數
risk_stats['總收容人數'] = risk_stats['總收容人數'].astype(int)

print("各風險等級統計:")
print(risk_stats.to_string(index=False))

# 計算風險覆蓋率
total_shelters = len(shelters_with_township)
risk_stats['百分比'] = (risk_stats['避難所數量'] / total_shelters * 100).round(1)

print(f"\n風險覆蓋率:")
for _, row in risk_stats.iterrows():
    print(f"{row['風險等級']}: {row['避難所數量']}個 ({row['百分比']}%)")

# 輸出第二個檔案
risk_output = os.path.join(output_folder, '風險等級統計.csv')
risk_stats.to_csv(risk_output, index=False, encoding='utf-8-sig')
print(f"✓ 已輸出風險等級統計檔案: {risk_output}")

print(f"\n=== 分析完成總結 ===")
print(f"✓ 已產生 2 個檔案:")
print(f"  1. {county_output} - 各縣市避難所和收容人數統計（包含各風險等級詳細資料）")
print(f"  2. {risk_output} - 各風險等級避難所數量統計")
print(f"✓ 總共分析了 {len(shelters_with_township)} 筆避難所資料")

# 7. 缺口判斷-風險最高的 Top 10 行政區排名分析

### 目標：找出收容量相對不足、風險最高的前 10 名行政區

1. 從 `.env` 讀取各縣市人口資料
2. 檢查並整理避難所收容人數欄位
3. 以空間連接判定各避難所所屬縣市與鄉鎮市區
4. 統計各縣市不同風險等級的避難所收容人數
5. 計算各縣市加權風險收容人數與具體可行收容人數
6. 以具體可行收容人數占總人口比例作為風險評分
7. 排序後擷取風險最高的前 10 名行政區
8. 輸出排名分析結果為 CSV 檔案

### 計算方式

1. 加權風險收容人數  
   = 低風險收容人數 +（中風險收容人數 × 0.5）+（高風險收容人數 × 0.25）

2. 具體可行收容人數  
   = 安全區收容人數 +（加權風險收容人數 × 0.8）

3. 具體可行收容人數佔總人口百分比  
   =（具體可行收容人數 ÷ 總人口數）× 100

4. 風險評分  
   以具體可行收容人數佔總人口百分比作為判斷依據，百分比越低表示風險越高

### 輸出結果

輸出 `風險最高Top10行政區排名.csv`，內容包含前 10 名高風險行政區之排名、總人口數、安全區收容人數、加權過後風險區總收容人數、具體可行收容人數，以及相關百分比指標。

In [ ]:
# 風險最高的 Top 10 行政區排名分析

print("=== 風險最高的 Top 10 行政區排名分析 ===")

# 導入必要的套件
import pandas as pd
import geopandas as gpd
from dotenv import load_dotenv
import os

# 讀取 .env 檔案中的目標縣市和人口資料
load_dotenv()

# 讀取各縣市人口數量 - 處理可能的變數名稱
city_populations = {}

# 處理台北市的兩種可能變數名
taipei_pop = os.getenv('NEW_TAIPEI_POP') or os.getenv('TAIPEI_POP') or '0'
city_populations['新北市'] = int(taipei_pop)

city_populations['新竹市'] = int(os.getenv('HSINCHU_POP', '0'))
city_populations['桃園市'] = int(os.getenv('TAOYUAN_POP', '0'))
city_populations['臺中市'] = int(os.getenv('TAICHUNG_POP', '0'))
city_populations['臺南市'] = int(os.getenv('TAINAN_POP', '0'))
city_populations['高雄市'] = int(os.getenv('KAOHSIUNG_POP', '0'))
city_populations['基隆市'] = int(os.getenv('KEELUNG_POP', '0'))
city_populations['新竹縣'] = int(os.getenv('HSINCHU_COUNTY_POP', '0'))
city_populations['苗栗縣'] = int(os.getenv('MIOLI_POP', '0'))
city_populations['彰化縣'] = int(os.getenv('CHANGHUA_POP', '0'))
city_populations['南投縣'] = int(os.getenv('NANTOU_POP', '0'))
city_populations['雲林縣'] = int(os.getenv('YUNLIN_POP', '0'))
city_populations['嘉義縣'] = int(os.getenv('CHIAYI_COUNTY_POP', '0'))
city_populations['嘉義市'] = int(os.getenv('CHIAYI_CITY_POP', '0'))
city_populations['屏東縣'] = int(os.getenv('PINGTUNG_POP', '0'))
city_populations['宜蘭縣'] = int(os.getenv('ILAN_POP', '0'))
city_populations['花蓮縣'] = int(os.getenv('HUALIEN_POP', '0'))
city_populations['臺東縣'] = int(os.getenv('TAITUNG_POP', '0'))
city_populations['澎湖縣'] = int(os.getenv('PENGHU_POP', '0'))
city_populations['金門縣'] = int(os.getenv('KINMEN_POP', '0'))
city_populations['連江縣'] = int(os.getenv('LIENCHIANG_POP', '0'))

print(f"已讀取 {len(city_populations)} 個縣市的人口資料")

# 檢查收容人數欄位
print("\n檢查避難所資料欄位...")

# 找出收容人數相關欄位
capacity_cols = []
for col in shelters_with_risk.columns:
    col_lower = str(col).lower()
    if any(keyword in col_lower for keyword in ['收容人數', '容納人數', '人數', 'capacity']) and '地址' not in col and '名稱' not in col:
        capacity_cols.append(col)



if not capacity_cols:
    print("⚠ 未找到收容人數欄位，使用預設值 100")
    capacity_col = '收容人數'
    shelters_with_risk[capacity_col] = 100
else:
    capacity_col = capacity_cols[0]
    print(f"✓ 使用收容人數欄位: {capacity_col}")

# 處理收容人數資料
shelters_with_risk_clean = shelters_with_risk.copy()
shelters_with_risk_clean[capacity_col] = pd.to_numeric(shelters_with_risk_clean[capacity_col], errors='coerce')
shelters_with_risk_clean[capacity_col] = shelters_with_risk_clean[capacity_col].fillna(shelters_with_risk_clean[capacity_col].median()).astype(int)

# 按鄉鎮市區進行空間連接
shelters_with_township = gpd.sjoin(shelters_with_risk_clean, townships, how='left', predicate='within')

# 確定正確的鄉鎮欄位名稱
township_cols = [col for col in shelters_with_township.columns if any(keyword in col.upper() for keyword in ['TOWN', 'COUNTY'])]
town_name_col = None
county_name_col = None

for col in township_cols:
    if 'TOWN' in col.upper() and 'NAME' in col.upper() and 'right' not in col.lower():
        town_name_col = col
    elif 'COUNTY' in col.upper() and 'NAME' in col.upper() and 'right' not in col.lower():
        county_name_col = col

if town_name_col != 'TOWNNAME':
    shelters_with_township['TOWNNAME'] = shelters_with_township[town_name_col]
if county_name_col != 'COUNTYNAME':
    shelters_with_township['COUNTYNAME'] = shelters_with_township[county_name_col]

# 移除無法匹配的記錄
shelters_with_township = shelters_with_township.dropna(subset=['TOWNNAME'])
print(f"匹配後避難所數量: {len(shelters_with_township)} 筆")

# === 計算各縣市風險指標 ===
print(f"\n=== 計算各縣市風險指標 ===")

county_risk_analysis = []

for county in city_populations.keys():
    # 取得該縣市的避難所資料
    county_shelters = shelters_with_township[shelters_with_township['COUNTYNAME'] == county]
    
    if len(county_shelters) == 0:
        print(f"⚠ {county} 沒有避難所資料")
        continue
    
    # 計算各風險等級的收容人數
    high_risk_capacity = county_shelters[county_shelters['risk_level'] == '高'][capacity_col].sum()
    med_risk_capacity = county_shelters[county_shelters['risk_level'] == '中'][capacity_col].sum()
    low_risk_capacity = county_shelters[county_shelters['risk_level'] == '低'][capacity_col].sum()
    safe_capacity = county_shelters[county_shelters['risk_level'] == '安全'][capacity_col].sum()
    
    # 取得總人口數
    total_population = city_populations.get(county, 0)
    
    # 計算加權過後風險區收容人數
    # 加權方法：低風險收容人數 + 中風險 * 0.5 + 高風險 * 0.25
    weighted_risk_capacity = low_risk_capacity + (med_risk_capacity * 0.5) + (high_risk_capacity * 0.25)
    
    # 計算具體可行收容人數
    # 公式：安全區收容人數 + (加權風險區總收容人數 × 0.8)
    feasible_capacity = safe_capacity + (weighted_risk_capacity * 0.8)
    
    # 計算具體可行收容人數佔總人口百分比
    if total_population > 0:
        feasible_capacity_percentage = (feasible_capacity / total_population) * 100
    else:
        feasible_capacity_percentage = 0
    
    # 計算安全區收容人數佔總人口百分比
    if total_population > 0:
        safe_capacity_percentage = (safe_capacity / total_population) * 100
    else:
        safe_capacity_percentage = 0
    
    # 儲存分析結果
    county_data = {
        '縣市名稱': county,
        '總人口數': total_population,
        '安全區收容人數': safe_capacity,
        '加權風險收容人數': weighted_risk_capacity,
        '具體可行收容人數': feasible_capacity,
        '具體可行百分比': feasible_capacity_percentage,
        '安全區百分比': safe_capacity_percentage,
        '風險評分': feasible_capacity_percentage  # 風險高低根據具體可行百分比判斷（百分比越低風險越高）
    }
    
    county_risk_analysis.append(county_data)

# 轉換為 DataFrame
risk_df = pd.DataFrame(county_risk_analysis)

# 按風險評分排序（百分比越低風險越高）
risk_df_sorted = risk_df.sort_values('風險評分', ascending=True)

# 取前10名風險最高的行政區
top_10_risk = risk_df_sorted.head(10).copy()

# 加上排名欄位
top_10_risk = top_10_risk.reset_index(drop=True)
top_10_risk['名次'] = range(1, 11)

# 重新排列欄位順序
final_columns = [
    '名次',
    '縣市名稱', 
    '總人口數',
    '安全區收容人數',
    '加權風險收容人數',
    '具體可行收容人數',
    '具體可行百分比',
    '安全區百分比'
]

top_10_risk_final = top_10_risk[final_columns]

# 重新命名欄位
top_10_risk_final.columns = [
    '名次',
    '縣市',
    '總人口數',
    '安全區收容人數',
    '加權過後風險區總收容人數',
    '具體可行收容人數',
    '具體可行收容人數佔總人口的百分比',
    '安全區收容人數佔總人口的百分比'
]

# 格式化數值顯示
top_10_risk_final['總人口數'] = top_10_risk_final['總人口數'].apply(lambda x: f"{x:,}")
top_10_risk_final['安全區收容人數'] = top_10_risk_final['安全區收容人數'].apply(lambda x: f"{x:,}")
top_10_risk_final['加權過後風險區總收容人數'] = top_10_risk_final['加權過後風險區總收容人數'].apply(lambda x: f"{x:,.0f}")
top_10_risk_final['具體可行收容人數'] = top_10_risk_final['具體可行收容人數'].apply(lambda x: f"{x:,.0f}")
top_10_risk_final['具體可行收容人數佔總人口的百分比'] = top_10_risk_final['具體可行收容人數佔總人口的百分比'].apply(lambda x: f"{x:.2f}%")
top_10_risk_final['安全區收容人數佔總人口的百分比'] = top_10_risk_final['安全區收容人數佔總人口的百分比'].apply(lambda x: f"{x:.2f}%")

# 輸出檔案
output_filename = os.path.join(output_folder, '風險最高Top10行政區排名.csv')
top_10_risk_final.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"\n✓ 已輸出風險排名檔案: {output_filename}")

print(f"\n=== 分析完成總結 ===")
print(f"✓ 分析了 {len(risk_df)} 個縣市")
print(f"✓ 產生風險最高 Top 10 行政區排名")
print(f"✓ 包含8個完整欄位的分析報告")
print(f"✓ 輸出檔案: {output_filename}")

# 8. 建立避難所風險程度互動式地圖
---
### 目標：建立避難所風險程度互動式地圖，呈現河川、鄉鎮界線與避難所風險分布情形

1. 將河川、避難所與鄉鎮界線資料轉換為 WGS84 坐標系（EPSG:4326），以利網頁地圖顯示
2. 以河川資料中心點計算地圖顯示中心位置
3. 建立互動式地圖底圖
4. 加入鄉鎮界線圖層作為背景
5. 加入河川圖層顯示河道分布
6. 加入避難所圖層，並依風險等級以不同顏色標記
7. 建立圖例與圖層控制功能
8. 輸出互動式 HTML 地圖檔案

### 圖層說明

本互動式地圖共包含 3 個主要圖層：

1. 鄉鎮界線圖層  
   作為背景參考，用以顯示各鄉鎮市區範圍

2. 河川圖層  
   以藍色顯示河流空間位置

3. 避難所圖層  
   以群聚標記方式呈現所有避難所，並依風險等級進行顏色分類：
   - 紅色：高風險
   - 橘色：中風險
   - 黃色：低風險
   - 綠色：安全

### 地圖操作說明

1. 可使用滑鼠滾輪進行地圖縮放
2. 按住滑鼠左鍵拖曳可移動地圖位置
3. 點擊避難所標記可查看詳細資訊，包括：
   - 避難收容處所名稱
   - 收容人數
   - 風險等級
4. 右上角圖層控制可開啟或關閉各圖層顯示
5. 避難所數量較密集時會以群聚方式顯示，點擊群聚圓形可逐步放大查看個別避難所位置

### 輸出結果

輸出 `避難所風險程度互動式地圖.html`，可於瀏覽器中開啟並操作互動式地圖。

### 地圖特色

1. 以鄉鎮界線作為空間背景，便於判讀避難所所在行政區位置
2. 以河川圖層輔助辨識與河道之空間關係
3. 避難所依風險等級分色顯示，可快速辨識高、中、低風險及安全避難所
4. 提供圖例與圖層控制，提升地圖判讀與操作便利性

In [ ]:
import folium
from folium import plugins

# 1. 轉換坐標系為 WGS84 用於顯示
print("轉換坐標系...")
rivers_wgs84 = rivers.to_crs('EPSG:4326')
shelters_wgs84 = shelters_with_risk.to_crs('EPSG:4326')
townships_wgs84 = townships.to_crs('EPSG:4326')
print("✓ 坐標系轉換完成")

# 2. 計算地圖中心點
river_centroids = rivers_wgs84.geometry.centroid
center_lat = river_centroids.y.mean()
center_lon = river_centroids.x.mean()
print(f"地圖中心: ({center_lat:.6f}, {center_lon:.6f})")

# 3. 建立互動式地圖
print("\n建立互動式地圖...")
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    tiles='OpenStreetMap'
)

# 4. 添加鄉鎮界線 (背景)
folium.GeoJson(
    townships_wgs84,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': 'gray',
        'weight': 1,
        'fillOpacity': 0.1
    },
    name='鄉鎮界線'
).add_to(m)
print("✓ 鄉鎮界線添加完成")

# 5. 添加河流 (藍色)
folium.GeoJson(
    rivers_wgs84,
    style_function=lambda x: {
        'fillColor': '#2196F3',
        'color': '#1976D2',
        'weight': 2,
        'fillOpacity': 0.6
    },
    name='河流'
).add_to(m)
print(f"✓ 河流: {len(rivers_wgs84)} 條")

# 6. 添加避難所 (依風險等級著色)
risk_colors = {
    '高': '#F44336',  # 紅色
    '中': '#FF9800',  # 橙色
    '低': '#FFEB3B',  # 黃色
    '安全': '#4CAF50'  # 綠色
}

# 創建淡紫色標記群組（無邊框）
marker_cluster = plugins.MarkerCluster(
    name='避難所群組',
    icon_create_function='''
    function(cluster) {
        return new L.DivIcon({
            html: '<div style="background-color: rgba(206, 147, 216, 0.6); color: #4A148C; border-radius: 50%; width: 35px; height: 35px; display: flex; align-items: center; justify-content: center; font-weight: bold;">' + cluster.getChildCount() + '</div>',
            className: 'custom-cluster-icon',
            iconSize: [35, 35]
        });
    }
    '''
)

print(f"處理 {len(shelters_wgs84)} 個避難所...")

# 處理避難所標記
processed_count = 0
for idx, shelter in shelters_wgs84.iterrows():
    try:
        lat, lon = shelter.geometry.y, shelter.geometry.x
        risk_level = shelter['risk_level']
        color = risk_colors.get(risk_level, '#808080')
        capacity = shelter.get('預計收容人數', 'N/A')
        name = shelter.get('避難收容處所名稱', '未知')
        
        # 建立彈出視窗內容
        popup_content = f"""
        <div style="font-family: Arial, sans-serif; min-width: 200px;">
            <h4 style="color: {color}; margin: 5px 0;">{name}</h4>
            <p style="margin: 3px 0;"><strong>收容人數:</strong> {capacity} 人</p>
            <p style="margin: 3px 0;"><strong>風險等級:</strong> <span style="color: {color}; font-weight: bold;">{risk_level}</span></p>
        </div>
        """
        
        # 建立標記
        folium.CircleMarker(
            location=[lat, lon],
            radius=6,
            popup=folium.Popup(popup_content, max_width=200),
            tooltip=f"{name}<br>風險: {risk_level}<br>收容: {capacity}人",
            color=color,
            fillColor=color,
            fillOpacity=0.8,
            weight=2
        ).add_to(marker_cluster)
        
        processed_count += 1
    except Exception as e:
        continue

# 將標記群組添加到地圖
marker_cluster.add_to(m)
print(f"✓ 避難所: {processed_count} 個")

# 7. 添加圖例
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; width: 220px; height: 240px; 
            background-color: white; border: 2px solid grey; z-index: 9999; 
            font-family: Arial, sans-serif; font-size: 12px; padding: 10px; border-radius: 5px;">
    <h4 style="margin: 0 0 10px 0; text-align: center;">風險地圖圖例</h4>
    
    <h5 style="margin: 10px 0 5px 0; color: #1976D2;">河流</h5>
    <div style="background: #2196F3; height: 15px; margin: 2px 0; border: 1px solid #1976D2;"></div>
    
    <h5 style="margin: 10px 0 5px 0;">避難所風險等級</h5>
    <div style="display: flex; align-items: center; margin: 3px 0;">
        <div style="width: 15px; height: 15px; background: #F44336; border-radius: 50%; margin-right: 8px;"></div>
        <span>高風險</span>
    </div>
    <div style="display: flex; align-items: center; margin: 3px 0;">
        <div style="width: 15px; height: 15px; background: #FF9800; border-radius: 50%; margin-right: 8px;"></div>
        <span>中風險</span>
    </div>
    <div style="display: flex; align-items: center; margin: 3px 0;">
        <div style="width: 15px; height: 15px; background: #FFEB3B; border-radius: 50%; margin-right: 8px;"></div>
        <span>低風險</span>
    </div>
    <div style="display: flex; align-items: center; margin: 3px 0;">
        <div style="width: 15px; height: 15px; background: #4CAF50; border-radius: 50%; margin-right: 8px;"></div>
        <span>安全</span>
    </div>
    
    <div style="margin-top: 10px; font-size: 11px; color: #666;">
        💡 點擊避難所查看詳細資訊
    </div>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(m)

# 8. 顯示統計資訊
risk_counts = shelters_with_risk['risk_level'].value_counts()
print(f"\n=== 避難所風險統計 ===")
print(f"高風險: {risk_counts.get('高', 0)} 個")
print(f"中風險: {risk_counts.get('中', 0)} 個")
print(f"低風險: {risk_counts.get('低', 0)} 個")
print(f"安全: {risk_counts.get('安全', 0)} 個")

# 儲存地圖
output_file = os.path.join(output_folder, '避難所風險程度互動式地圖.html')
m.save(output_file)
print(f"\n✅ 地圖完成！")
print(f"📁 已儲存: {output_file}")
print(f"🎯 地圖特色:")
print(f"   • 河川面（藍色）")
print(f"   • 避難所依風險等級著色：紅=高風險、橙=中風險、黃=低風險、綠=安全")
print(f"   • 鄉鎮界作為背景")
print(f"   • 點擊避難所可顯示名稱、收容人數、風險等級")

# 9. 製作 Top 10 高風險行政區視覺化分析
---
### 目標：比較風險最高前 10 名行政區的總避難所數量與總收容量差異

1. 讀取 `風險最高Top10行政區排名.csv`
2. 讀取 `各縣市避難所統計.csv`
3. 篩選出前 10 名高風險行政區的統計資料
4. 依具體可行收容人數佔總人口百分比排序行政區
5. 建立雙軸長條圖，比較各行政區之總避難所數量與總收容量
6. 在圖中標示各長條對應數值
7. 輸出統計圖為 PNG 檔案

### 視覺化說明

本圖採用雙軸長條圖呈現前 10 名高風險行政區的避難資源配置情形：

1. 左側 Y 軸表示總避難所數量（個）
2. 右側 Y 軸表示總收容量（人）
3. X 軸為風險最高之前 10 名行政區
4. 各行政區依具體可行收容人數佔總人口百分比由低至高排序，百分比越低代表風險越高

### 圖表內容

1. 紅色長條表示總避難所數量
2. 藍綠色長條表示總收容量
3. 每個長條上方皆標示實際數值
4. 圖中包含圖例與網格線，以利比較不同行政區之差異

### 輸出結果

輸出 `risk_map.png`，內容為 Top 10 高風險行政區之總避難所數量與總收容量比較圖。

### 分析用途

1. 比較高風險行政區之避難所設置數量差異
2. 比較高風險行政區之總收容量差異
3. 輔助判斷行政區避難資源是否充足
4. 作為後續風險評估與資源配置之視覺化依據

In [ ]:
# 長條圖：Top 10 高風險行政區的總避難所數量 vs. 總收容量
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 設定中文字體
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 1. 讀取風險最高Top10行政區排名資料
risk_top10 = pd.read_csv('output\\風險最高Top10行政區排名.csv')

# 2. 讀取各縣市避難所統計資料來獲取總避難所數量和總收容量
county_stats = pd.read_csv('output\\各縣市避難所統計.csv')

# 3. 合併資料，只取前10名高風險行政區的總資料
top_10_counties = risk_top10['縣市'].tolist()
top_10_data = county_stats[county_stats['縣市名稱'].isin(top_10_counties)].copy()

# 4. 按具體可行收容人數佔總人口的百分比排序（百分比越低，風險越高）
# 從 risk_top10 中獲取排序順序
risk_order = risk_top10.sort_values('具體可行收容人數佔總人口的百分比')['縣市'].tolist()
top_10_data['縣市名稱'] = pd.Categorical(top_10_data['縣市名稱'], categories=risk_order, ordered=True)
top_10_data = top_10_data.sort_values('縣市名稱')

print("=== Top 10 高風險行政區（總避難所數量 vs. 總收容量）===")
for idx, county in enumerate(risk_order, 1):
    county_data = top_10_data[top_10_data['縣市名稱'] == county]
    if len(county_data) > 0:
        row = county_data.iloc[0]
        percentage = risk_top10[risk_top10['縣市'] == county]['具體可行收容人數佔總人口的百分比'].iloc[0]
        print(f"{idx}. {county} - 具體可行收容佔比: {percentage}, 總避難所: {row['避難所數量']}個, 總收容量: {row['總收容人數']:,}人")

# 5. 創建雙軸長條圖
fig, ax1 = plt.subplots(figsize=(14, 8))

# 設定位置
x = np.arange(len(top_10_data))
width = 0.35

# 左軸：總避難所數量
bars1 = ax1.bar(x - width/2, top_10_data['避難所數量'], 
                 width, label='總避難所數量', color='#FF6B6B', alpha=0.8, edgecolor='#FF4757', linewidth=1.5)

# 設定左軸
ax1.set_xlabel('行政區', fontsize=12, labelpad=10, color='black')
ax1.set_ylabel('總避難所數量 (個)', fontsize=12, labelpad=10, color='black')
ax1.tick_params(axis='y', labelcolor='black')
ax1.tick_params(axis='x', labelcolor='black')

# 右軸：總收容量
ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, top_10_data['總收容人數'], 
                 width, label='總收容量', color='#4ECDC4', alpha=0.8, edgecolor='#00B894', linewidth=1.5)

# 設定右軸
ax2.set_ylabel('總收容量 (人)', fontsize=12, labelpad=10, color='black')
ax2.tick_params(axis='y', labelcolor='black')

# 設定X軸標籤
ax1.set_xticks(x)
ax1.set_xticklabels(top_10_data['縣市名稱'], rotation=45, ha='right')

# 添加數值標籤
# 左軸標籤（總避難所數量）
for i, bar in enumerate(bars1):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + max(top_10_data['避難所數量']) * 0.02,
             f'{int(height)}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#FF6B6B')

# 右軸標籤（總收容量）
for i, bar in enumerate(bars2):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + max(top_10_data['總收容人數']) * 0.01,
             f'{int(height):,}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#4ECDC4')

# 設定圖表樣式
plt.title('Top 10 高風險行政區：總避難所數量 vs. 總收容量', fontsize=18, fontweight='bold', pad=20, color='black')

# 添加圖例
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

# 設定網格
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# 調整邊距
plt.tight_layout()

# 儲存圖片
plt.savefig('risk_map.png', dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✅ 統計圖已儲存為 risk_map.png")

# 顯示圖表
plt.show()

# 輸出簡要統計資訊
print(f"\n=== 統計摘要 ===")
print(f"前10名高風險行政區總計:")
print(f"  • 總避難所: {top_10_data['避難所數量'].sum()} 個")
print(f"  • 總收容量: {top_10_data['總收容人數'].sum():,} 人")

# 生成 shelter_risk_audit.json 避難所風險清單
---
### 目標：將避難所風險標記結果轉換為 JSON 格式清單，作為後續查核與資料交換使用
1. 讀取已完成風險標記的 `shelter_marked.csv`
2. 擷取每筆避難所的基本資訊
3. 建立 JSON 格式的避難所風險清單
4. 輸出為 `shelter_risk_audit.json`
5. 檢查輸出結果與資料筆數是否正確

### JSON 欄位說明
每筆避難所資料包含以下欄位：

1. `shelter_id`：避難所編號
2. `name`：避難收容處所名稱
3. `risk_level`：風險等級
4. `capacity`：預計收容人數

### 處理流程說明
1. 由 `shelter_marked.csv` 逐筆讀取避難所資料
2. 將避難所名稱、風險等級與收容人數整理為結構化欄位
3. 以串列方式儲存所有避難所資料
4. 轉存為格式化 JSON 檔案
5. 顯示風險等級統計、前 5 筆資料範例與檔案格式確認資訊

### 輸出結果
輸出 `shelter_risk_audit.json`，內容為完整避難所風險清單，可供後續查核、交換或系統串接使用。

### 分析用途
1. 建立避難所風險資料的結構化清單
2. 便於後續系統讀取與資料交換
3. 提供風險等級查核與抽樣檢視依據
4. 作為後續視覺化、稽核或 API 串接之基礎資料

In [ ]:
# 生成 shelter_risk_audit.json - 避難所風險清單
import pandas as pd
import json

# 1. 讀取已標記風險等級的避難所資料
shelters_data = pd.read_csv('output/shelter_marked.csv')

# 2. 準備 JSON 資料
shelter_audit_list = []

for idx, row in shelters_data.iterrows():
    shelter_info = {
        "shelter_id": idx + 1,
        "name": row.get('避難收容處所名稱', '未知'),
        "risk_level": row.get('risk_level', '未知'),
        "capacity": row.get('預計收容人數', 0)
    }
    shelter_audit_list.append(shelter_info)

# 3. 儲存為 JSON 檔案
with open('shelter_risk_audit.json', 'w', encoding='utf-8') as f:
    json.dump(shelter_audit_list, f, ensure_ascii=False, indent=2)

print(f"✅ 已生成 shelter_risk_audit.json")
print(f"📊 總共 {len(shelter_audit_list)} 個避難所")

# 4. 顯示統計資訊
risk_counts = shelters_data['risk_level'].value_counts()
print(f"\n=== 風險等級統計 ===")
for risk_level, count in risk_counts.items():
    print(f"{risk_level}: {count} 個")

# 5. 顯示前5筆資料範例
print(f"\n=== 前5筆資料範例 ===")
for i, shelter in enumerate(shelter_audit_list[:5], 1):
    print(f"{i}. ID: {shelter['shelter_id']}, 名稱: {shelter['name']}, 風險: {shelter['risk_level']}, 收容: {shelter['capacity']}人")

# 6. 檢查 JSON 檔案內容
print(f"\n=== JSON 檔案格式確認 ===")
with open('shelter_risk_audit.json', 'r', encoding='utf-8') as f:
    content = f.read()
    print(f"檔案大小: {len(content)} 字元")
    print(f"前200字元: {content[:200]}...")